# 面试题：知识图谱怎样从数据建图，并真正支持多跳查询？

## 可以直接复述的回答

知识图谱的核心不是画一张关系图，而是给实体稳定 ID、给边明确方向与类型，并保存来源和时间等可审计属性。构图后应先用邻接表验证一跳关系，再实现受关系序列约束的多跳遍历，避免“路径存在但语义错误”。平铺关键词查询只能回答同一条记录里的直接事实，无法组合“订单—团队—负责人”这类问题。BFS 或按关系逐层扩展时必须记录 visited 与最大跳数，否则环会导致重复访问甚至无限循环。结果最好返回完整路径而不是只给终点，便于解释和追溯。本题用订单、商品、仓库、政策、团队、员工、服务和告警等实体构造 19 条三元组，并实际运行 6 个查询与环路失败案例。

## 真实案例

三元组模拟电商运营知识图谱的脱敏子图，实体采用“类型_业务键”形式避免同名碰撞。真实系统还会为边附加 source、valid_from 等属性；本教学版只保留关系机制，不能替代图数据库。

In [1]:
from collections import defaultdict, deque  # 导入邻接表与广度优先队列所需容器
from pprint import pprint  # 导入结构化打印函数以展示图和路径账本
triples = [("客户_林女士", "下单", "订单_A100"), ("订单_A100", "包含商品", "商品_降噪耳机"), ("商品_降噪耳机", "发货自", "仓库_华东"), ("商品_降噪耳机", "适用政策", "政策_数码七天"), ("订单_A100", "售后归属", "团队_售后"), ("团队_售后", "负责人", "员工_周岚"), ("文档_退款SOP", "维护方", "团队_售后"), ("订单_A100", "触发告警", "告警_履约延迟"), ("告警_履约延迟", "关联服务", "服务_订单"), ("服务_订单", "负责团队", "团队_平台"), ("团队_平台", "负责人", "员工_王磊"), ("客户_赵先生", "下单", "订单_B200"), ("订单_B200", "包含商品", "商品_机械键盘"), ("商品_机械键盘", "发货自", "仓库_华南"), ("商品_机械键盘", "适用政策", "政策_外设换新"), ("服务_订单", "依赖", "服务_支付"), ("服务_支付", "依赖", "服务_订单"), ("文档_退款SOP", "引用政策", "政策_数码七天"), ("员工_周岚", "属于", "团队_售后")]  # 构造十九条有业务语义且包含环的三元组
query_specs = [{"id": "Q1", "问题": "A100 包含什么商品", "start": "订单_A100", "relations": ["包含商品"], "expected": "商品_降噪耳机"}, {"id": "Q2", "问题": "林女士买了什么商品", "start": "客户_林女士", "relations": ["下单", "包含商品"], "expected": "商品_降噪耳机"}, {"id": "Q3", "问题": "降噪耳机从哪里发货", "start": "商品_降噪耳机", "relations": ["发货自"], "expected": "仓库_华东"}, {"id": "Q4", "问题": "A100 的售后负责人是谁", "start": "订单_A100", "relations": ["售后归属", "负责人"], "expected": "员工_周岚"}, {"id": "Q5", "问题": "退款SOP由谁负责", "start": "文档_退款SOP", "relations": ["维护方", "负责人"], "expected": "员工_周岚"}, {"id": "Q6", "问题": "履约延迟告警最终由谁负责", "start": "告警_履约延迟", "relations": ["关联服务", "负责团队", "负责人"], "expected": "员工_王磊"}]  # 定义六个一跳与多跳混合查询
print("三元组输入预览：")  # 输出真实图数据标题
pprint(triples)  # 展示全部实体关系事实
print("待回答的图查询：")  # 输出查询规格标题
pprint(query_specs)  # 展示起点、关系序列和人工期望终点

三元组输入预览：
[('客户_林女士', '下单', '订单_A100'),
 ('订单_A100', '包含商品', '商品_降噪耳机'),
 ('商品_降噪耳机', '发货自', '仓库_华东'),
 ('商品_降噪耳机', '适用政策', '政策_数码七天'),
 ('订单_A100', '售后归属', '团队_售后'),
 ('团队_售后', '负责人', '员工_周岚'),
 ('文档_退款SOP', '维护方', '团队_售后'),
 ('订单_A100', '触发告警', '告警_履约延迟'),
 ('告警_履约延迟', '关联服务', '服务_订单'),
 ('服务_订单', '负责团队', '团队_平台'),
 ('团队_平台', '负责人', '员工_王磊'),
 ('客户_赵先生', '下单', '订单_B200'),
 ('订单_B200', '包含商品', '商品_机械键盘'),
 ('商品_机械键盘', '发货自', '仓库_华南'),
 ('商品_机械键盘', '适用政策', '政策_外设换新'),
 ('服务_订单', '依赖', '服务_支付'),
 ('服务_支付', '依赖', '服务_订单'),
 ('文档_退款SOP', '引用政策', '政策_数码七天'),
 ('员工_周岚', '属于', '团队_售后')]
待回答的图查询：
[{'expected': '商品_降噪耳机',
  'id': 'Q1',
  'relations': ['包含商品'],
  'start': '订单_A100',
  '问题': 'A100 包含什么商品'},
 {'expected': '商品_降噪耳机',
  'id': 'Q2',
  'relations': ['下单', '包含商品'],
  'start': '客户_林女士',
  '问题': '林女士买了什么商品'},
 {'expected': '仓库_华东',
  'id': 'Q3',
  'relations': ['发货自'],
  'start': '商品_降噪耳机',
  '问题': '降噪耳机从哪里发货'},
 {'expected': '员工_周岚',
  'id': 'Q4',
  'relations': ['售后归属', '负责人'],
  'star

## Baseline / 基线：只查一条直接边

平铺索引可以根据起点和第一种关系找到邻居，却不会继续沿关系组合事实。因此它能回答商品和仓库的一跳问题，但会把订单、团队等中间节点误当最终答案。

In [2]:
def direct_lookup(start, relation):  # 定义只能读取单条直接关系的基线
    matches = [target for source, edge, target in triples if source == start and edge == relation]  # 扫描三元组找一跳终点
    return matches[0] if matches else None  # 返回第一个直接邻居或空值
baseline_rows = []  # 创建逐查询基线结果表
for spec in query_specs:  # 遍历六个图查询
    prediction = direct_lookup(spec["start"], spec["relations"][0])  # 只执行关系序列的第一跳
    baseline_rows.append({"问题": spec["问题"], "期望": spec["expected"], "一跳结果": prediction, "正确": prediction == spec["expected"]})  # 保存基线决策和正确性
baseline_hits = sum(row["正确"] for row in baseline_rows)  # 统计一跳基线命中数
print("直接边 Baseline：")  # 输出基线结果标题
pprint(baseline_rows)  # 展示多跳问题为何只得到中间节点
print(f"Baseline 命中：{baseline_hits}/{len(query_specs)}")  # 输出基线汇总指标

直接边 Baseline：
[{'一跳结果': '商品_降噪耳机', '期望': '商品_降噪耳机', '正确': True, '问题': 'A100 包含什么商品'},
 {'一跳结果': '订单_A100', '期望': '商品_降噪耳机', '正确': False, '问题': '林女士买了什么商品'},
 {'一跳结果': '仓库_华东', '期望': '仓库_华东', '正确': True, '问题': '降噪耳机从哪里发货'},
 {'一跳结果': '团队_售后', '期望': '员工_周岚', '正确': False, '问题': 'A100 的售后负责人是谁'},
 {'一跳结果': '团队_售后', '期望': '员工_周岚', '正确': False, '问题': '退款SOP由谁负责'},
 {'一跳结果': '服务_订单', '期望': '员工_王磊', '正确': False, '问题': '履约延迟告警最终由谁负责'}]
Baseline 命中：2/6


## 构图：有向带类型邻接表

邻接表的每条元素同时保留 relation 与 target。仅保存 target 会丢掉边语义，导致“负责人”和“属于”等不同关系被混在一起。

In [3]:
adjacency = defaultdict(list)  # 创建 source 到有类型出边列表的邻接表
reverse_adjacency = defaultdict(list)  # 创建 target 到反向入边列表的索引
for source, relation, target in triples:  # 遍历十九条原始三元组
    adjacency[source].append((relation, target))  # 保存有向关系和终点
    reverse_adjacency[target].append((relation, source))  # 保存反向关系以支持溯源查询
for source in adjacency:  # 遍历每个有出边的实体
    adjacency[source] = sorted(adjacency[source])  # 固定邻接边顺序保证路径结果稳定
node_set = {node for triple in triples for node in (triple[0], triple[2])}  # 收集图中的全部实体 ID
print(f"图统计：nodes={len(node_set)}, edges={len(triples)}")  # 输出图规模
print("订单_A100 的出边：", adjacency["订单_A100"])  # 展示一个实体的带类型邻接表
print("团队_售后的入边：", reverse_adjacency["团队_售后"])  # 展示反向索引如何支持来源追踪

图统计：nodes=18, edges=19
订单_A100 的出边： [('包含商品', '商品_降噪耳机'), ('售后归属', '团队_售后'), ('触发告警', '告警_履约延迟')]
团队_售后的入边： [('售后归属', '订单_A100'), ('维护方', '文档_退款SOP'), ('属于', '员工_周岚')]


## 手写关系约束遍历与中间 frontier

每一层只沿当前指定 relation 扩展，并为每个候选保留完整“实体—关系—实体”路径。这样即使图中存在更短但语义不对的边，也不会误答。

In [4]:
def follow_relation_path(start, relations):  # 定义按关系序列执行的多跳遍历
    frontier = [(start, [start])]  # 初始化当前实体和可解释路径
    traversal_ledger = [{"step": 0, "relation": None, "frontier": [start]}]  # 记录起点 frontier
    for step, expected_relation in enumerate(relations, start=1):  # 逐层消费查询指定的关系类型
        next_frontier = []  # 创建下一层候选列表
        for current, path in frontier:  # 遍历当前层所有实体及其路径
            for relation, target in adjacency.get(current, []):  # 遍历当前实体的全部有类型出边
                if relation == expected_relation:  # 只扩展语义与查询约束一致的边
                    next_frontier.append((target, path + [relation, target]))  # 保存终点及完整可读路径
        frontier = next_frontier  # 把下一层候选设为新的 frontier
        traversal_ledger.append({"step": step, "relation": expected_relation, "frontier": [node for node, _ in frontier]})  # 保存本层关系和候选实体
    return frontier, traversal_ledger  # 返回最终候选及逐层访问账本
example_frontier, example_ledger = follow_relation_path(query_specs[-1]["start"], query_specs[-1]["relations"])  # 实际执行三跳告警负责人查询
print("Q6 的逐层 frontier 与关系约束：")  # 输出关键中间过程标题
pprint(example_ledger)  # 展示告警到服务、团队和员工的每一步候选
print("Q6 完整路径：", example_frontier[0][1])  # 展示可审计的实体关系路径

Q6 的逐层 frontier 与关系约束：
[{'frontier': ['告警_履约延迟'], 'relation': None, 'step': 0},
 {'frontier': ['服务_订单'], 'relation': '关联服务', 'step': 1},
 {'frontier': ['团队_平台'], 'relation': '负责团队', 'step': 2},
 {'frontier': ['员工_王磊'], 'relation': '负责人', 'step': 3}]
Q6 完整路径： ['告警_履约延迟', '关联服务', '服务_订单', '负责团队', '团队_平台', '负责人', '员工_王磊']


## 结果表与结果解读

关系约束遍历对一跳和多跳使用同一接口，并返回完整路径。相比只查第一条边，它能把中间节点继续展开到真正答案；路径还能解释为什么命中该员工，而不是只给一个无法审计的名字。

In [5]:
graph_rows = []  # 创建逐查询图推理结果表
for spec in query_specs:  # 遍历六个查询规格
    frontier, ledger = follow_relation_path(spec["start"], spec["relations"])  # 按关系约束实际运行多跳查询
    prediction = frontier[0][0] if frontier else None  # 读取首个确定性终点
    path = frontier[0][1] if frontier else []  # 读取对应完整关系路径
    graph_rows.append({"问题": spec["问题"], "期望": spec["expected"], "预测": prediction, "路径": path, "访问层数": len(ledger) - 1, "正确": prediction == spec["expected"]})  # 保存逐样本结果和解释
graph_hits = sum(row["正确"] for row in graph_rows)  # 统计多跳实现命中数
print("逐查询路径结果：")  # 输出结果表标题
pprint(graph_rows)  # 展示每个问题的终点、完整路径和层数
print(f"命中从 {baseline_hits}/{len(query_specs)} 提升到 {graph_hits}/{len(query_specs)}")  # 输出同数据对照指标

逐查询路径结果：
[{'期望': '商品_降噪耳机',
  '正确': True,
  '访问层数': 1,
  '路径': ['订单_A100', '包含商品', '商品_降噪耳机'],
  '问题': 'A100 包含什么商品',
  '预测': '商品_降噪耳机'},
 {'期望': '商品_降噪耳机',
  '正确': True,
  '访问层数': 2,
  '路径': ['客户_林女士', '下单', '订单_A100', '包含商品', '商品_降噪耳机'],
  '问题': '林女士买了什么商品',
  '预测': '商品_降噪耳机'},
 {'期望': '仓库_华东',
  '正确': True,
  '访问层数': 1,
  '路径': ['商品_降噪耳机', '发货自', '仓库_华东'],
  '问题': '降噪耳机从哪里发货',
  '预测': '仓库_华东'},
 {'期望': '员工_周岚',
  '正确': True,
  '访问层数': 2,
  '路径': ['订单_A100', '售后归属', '团队_售后', '负责人', '员工_周岚'],
  '问题': 'A100 的售后负责人是谁',
  '预测': '员工_周岚'},
 {'期望': '员工_周岚',
  '正确': True,
  '访问层数': 2,
  '路径': ['文档_退款SOP', '维护方', '团队_售后', '负责人', '员工_周岚'],
  '问题': '退款SOP由谁负责',
  '预测': '员工_周岚'},
 {'期望': '员工_王磊',
  '正确': True,
  '访问层数': 3,
  '路径': ['告警_履约延迟', '关联服务', '服务_订单', '负责团队', '团队_平台', '负责人', '员工_王磊'],
  '问题': '履约延迟告警最终由谁负责',
  '预测': '员工_王磊'}]
命中从 2/6 提升到 6/6


## 失败案例：环路导致重复访问

服务_订单 与 服务_支付互相依赖。若无 visited 集合，按第一条“依赖”边不断走会在两个节点之间来回；下面先复现 6 步重复轨迹，再用带 visited 的 BFS 在有限访问后停止。

In [6]:
def naive_dependency_walk(start, steps):  # 定义不记录访问状态的错误遍历
    current = start  # 从给定服务节点开始
    trace = [current]  # 保存重复轨迹用于观察失败
    for _ in range(steps):  # 按固定步数模拟本可能无限的循环
        dependencies = [target for relation, target in adjacency[current] if relation == "依赖"]  # 获取当前服务的依赖边
        current = dependencies[0] if dependencies else current  # 没有 visited 时总选第一个依赖
        trace.append(current)  # 追加下一节点并暴露重复访问
    return trace  # 返回有环的错误轨迹
def safe_dependency_bfs(start, max_hops=10):  # 定义带 visited 和跳数上限的安全 BFS
    queue = deque([(start, 0)])  # 初始化待访问队列及深度
    visited = {start}  # 把起点加入已访问集合
    visit_order = []  # 保存每个节点仅一次的访问顺序
    while queue:  # 持续处理有限队列
        current, depth = queue.popleft()  # 取出队首节点和当前深度
        visit_order.append(current)  # 记录本次唯一访问
        if depth >= max_hops:  # 达到安全跳数上限时不再扩展
            continue  # 跳过当前节点的后继
        for relation, target in adjacency[current]:  # 遍历当前节点的有向边
            if relation == "依赖" and target not in visited:  # 只扩展未访问过的依赖节点
                visited.add(target)  # 入队前立即标记避免重复加入
                queue.append((target, depth + 1))  # 把新依赖及深度加入队尾
    return visit_order  # 返回无重复的有限访问顺序
unsafe_trace = naive_dependency_walk("服务_订单", 6)  # 复现两个服务间的来回循环
safe_trace = safe_dependency_bfs("服务_订单")  # 用 visited 与 max_hops 修正遍历
print("失败案例的重复轨迹：", unsafe_trace)  # 展示无 visited 时的环路
print("修正后的唯一访问顺序：", safe_trace)  # 展示安全 BFS 的有限结果

失败案例的重复轨迹： ['服务_订单', '服务_支付', '服务_订单', '服务_支付', '服务_订单', '服务_支付', '服务_订单']
修正后的唯一访问顺序： ['服务_订单', '服务_支付']


## 生产差距

线上知识图谱还要做实体消歧、schema 约束、边来源与有效期、租户 ACL、增量写入和冲突解决。大图查询应交给图数据库或分布式存储，并限制路径模式、跳数和结果规模；答案侧要回传路径证据并监控孤立节点、重复实体和过期边。

In [7]:
assert len(triples) == 19  # 验证真实案例包含十九条业务关系事实
assert baseline_hits == 2  # 验证一跳基线只能回答两个直接关系问题
assert graph_hits == len(query_specs)  # 验证关系约束遍历回答全部六个问题
assert example_frontier[0][0] == "员工_王磊"  # 验证三跳告警负责人查询到达正确员工
assert len(unsafe_trace) > len(set(unsafe_trace))  # 验证错误遍历确实重复访问环路节点
assert len(safe_trace) == len(set(safe_trace)) == 2  # 验证安全 BFS 对两个环节点各访问一次
print("最小回归测试通过：构图、多跳路径与环路保护均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：构图、多跳路径与环路保护均满足预期
